In [7]:
!pip install -q ultralytics mediapipe opencv-python

In [6]:
import cv2
import mediapipe as mp  # 스켈레톤을 추출하는 라이브러리
import numpy as np
import csv  # csv 저장을 위해 라이브러리 추가
import os   # 파일 경로 관리를 위해 라이브러리 추가

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence = 0.7,  # 감지 최소 신뢰도
    min_tracking_confidence = 0.3 # 추적 최소 신뢰도
)

# MediaPipe 그리기 유틸리티 초기화
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 경로
video_path = "aespa_test.mp4" # 분석할 동영상 파일 경로 입력
cap = cv2.VideoCapture(video_path)

# 출력 파일 이름 설정
output_filename = os.path.splitext(os.path.basename(video_path))[0] + "_skeleton.csv"
# CSV 파일 헤더 준비 (33개 랜드마크 * 4개 좌표)
landmarks = ['class'] + [f'{j}_{i}' for i in mp_pose.PoseLandmark._member_names_ for j in ('x', 'y', 'z', 'v')]

# 동영상 파일이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

print("스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.")

with open(output_filename, 'w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)  # 헤더 작성

    print(f"'{output_filename}' 파일에 스켈레톤 데이터 저장을 시작합니다.")
    frame_count = 0
    while cap.isOpened():
        # 동영상에서 프레임 읽기
        success, image = cap.read()

        if not success:
            print("동영상 스트림의 끝에 도달했거나 오류가 발생했습니다.")
            break

        # 성능 향상을 위해 이미지를 읽기 전용으로 표시
        image.flags.writeable = False
        # BGR 이미지를 RGB로 변환
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe Pose를 사용하여 포즈 감지 수행
        results = pose.process(image_rgb)

        # 이미지를 다시 쓰기 가능으로 변경
        image.flags.writeable = True

        # 감지된 스켈레톤(포즈 랜드마크)을 원본 이미지에 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec = mp_drawing.DrawingSpec(color = (245, 117, 66), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )

            # 랜드마크 데이터 추출 및 csv 행으로 변환
            try:
                # 'dance' 클래스로 분류 (필요에 따라 변경 가능)
                class_name = "dance"

                # 모든 랜드마크의 x, y, z, v 값을 순서대로 리스트에 담기
                pose_row = list(np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten())

                # 클래스 이름과 랜드마크 데이터를 합쳐서 한 행으로 만듦
                row = [class_name] + pose_row
                
                # csv 파일에 한 행 쓰기
                csv_writer.writerow(row)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류 발생: {e}")
                pass # 오류 발생 시 해당 프레임 건너뜀

        # 결과 영상 출력
        cv2.imshow('MediaPipe Pose Skeleton', image)

        frame_count += 1
        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# 자원 해제
cap.release()
cv2.destroyAllWindows()
pose.close()

print("스켈레톤 추출이 완료되었습니다.")

스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.
'aespa_test_skeleton.csv' 파일에 스켈레톤 데이터 저장을 시작합니다.
스켈레톤 추출이 완료되었습니다.


In [7]:
# mediapipe 라이브러리를 사용하여 기본 스켈레톤 추출
# YOLO 라이브러리를 사용하여, 다중 객체 탐지 (Top_down 방식)
# 칼만 필터(Kalman Filter)를 사용하여, 각 관절의 다음 위치를 예측하고 보정하여 성능 개선
# BoT-SORT 추적: 외모 특징까지 고려하는 강력한 추적기로 ID의 안정성을 극대화
# 상태 예측 및 보간: 추적을 잠시 놓친 스켈레톤의 위치를 예측하여, 시각적 끊김 없이 보이도록 처리
# 영상에서 탐지할 목표 인원 수를 사전에 입력 받아 성능 향상

import cv2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np
import random
import csv
from collections import Counter

# --------------------------------
# 1. 칼만 필터 및 추적 클래스 정의
# --------------------------------
class KalmanFilter:
    """A simple Kalman filter for 2D point tracking"""
    def __init__(self, dt=1, std_acc=1, x_std_meas=0.1, y_std_meas=0.1 ):
        # 상태 변수 [x, y, vx, vy] (위치, 속도)
        self.state = np.zeros((4, 1))
        # 상태 전이 행렬 (위치와 속도 관계 정의)
        self.F = np.array([[1, 0, dt, 0],
                           [0, 1, 0, dt],
                           [0, 0, 1, 0],
                           [0, 0, 0, 1]])
        # 측정 행렬 (상태 변수 중 위치만 측정)
        self.H = np.array([[1, 0, 0, 0],
                           [0, 1, 0, 0]])
        # 프로세스 노이즈 공분산 (모델의 불확실성)
        self.Q = np.eye(4)*std_acc**2
        # 측정 노이즈 공분산 (측정값의 불확실성)
        self.R = np.diag([x_std_meas**2, y_std_meas**2])
        # 오차 공분산 행렬
        self.P = np.eye(4)

    def predict(self):
        """Predict the next state."""
        self.state = np.dot(self.F, self.state)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.state
    
    def update(self, z):
        """Update the state with the new measurement."""
        # 칼만 이득 계산
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        # 상태 및 오차 공분산 업데이트
        self.state = self.state + np.dot(K, (z - np.dot(self.H, self.state)))
        self.P = self.P - np.dot(np.dot(K, self.H), self.P)
        return self.state

class TrackedPerson:
    """A class to store all data for a tracked person."""
    def __init__(self, track_id):
        self.id = track_id
        self.kalman_filters = [KalmanFilter() for _ in range(33)]
        self.landmarks = np.zeros((33, 4))
        self.disappeared_frames = 0
        self.color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

# ----------------
# 2. 초기 설정
# ----------------
# 목표 인원수를 직접 지정하거나, None으로 두어 자동 감지
MANUAL_PERSON_COUNT = 4

# YOLOv8 모델 로드 (가장 작고 빠른 'n' 모델 사용)
yolo_model = YOLO('yolov8m.pt')

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5)

# 동영상 파일 열기
video_path = "aespa_test.mp4"    # 스켈레톤 추출할 동영상 파일
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

# 결과 동영상을 저장하기 위한 설정
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
output_video_path = video_path.replace(".mp4", "_output.mp4")
out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

# CSV 파일 설정
csv_output_path = video_path.replace('.mp4', '_skeletons_data.csv')
csv_file = open(csv_output_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
# CSV 파일 헤더 작성
csv_header = ['frame', 'track_id', 'landmark_id'] + [f'{axis}_{i}' for i in range(33) for axis in ('x', 'y', 'z', 'visibility')]
# 더 유연한 Long Format 헤더로 변경
csv_header_long = ['frame', 'track_id', 'landmark_id', 'x', 'y', 'z', 'visibility']
csv_writer.writerow(csv_header_long)

tracked_skeletons = {} # 추적된 모든 사람의 데이터를 관리하는 딕셔너리
MAX_DISAPPEARED_FRAMES = 10 # 10프레임 동안 보이지 않으면 최종 삭제
frame_counter = 0 # 프레임 번호를 위한 카운터

print("스켈레톤 추출을 시작합니다. 결과는 동영상 파일로 저장됩니다.")

# --------------------------
# 3. 메인 루프: 프레임별 처리
# --------------------------
TARGET_PERSON_COUNT = MANUAL_PERSON_COUNT
if TARGET_PERSON_COUNT is None:
    print("목표 인원수 자동 감지를 시작합니다. (영상 시작 5초 분석)...")
    person_counts = []
    for i in range(int(fps * 5)): # 5초 분량의 프레임 분석
        success, frame = cap.read()
        if not success: break
        results = yolo_model(frame, classes=[0], conf=0.4, verbose=False)
        person_counts.append(len(results[0].boxes))
    
    if person_counts:
        TARGET_PERSON_COUNT = Counter(person_counts).most_common(1)[0][0]
        print(f"자동 감지 완료! 목표 인원수를 {TARGET_PERSON_COUNT}명으로 설정합니다.")
    else:
        TARGET_PERSON_COUNT = 1 # 감지 실패 시 기본값
        print("자동 감지 실패. 목표 인원수를 1명으로 설정합니다.")
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # 분석 후 비디오를 다시 처음으로 되돌림

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # STEP 1: 사람 탐지 (BoT-SORT 추적기 사용)
    # 현재 프레임에서 모든 객체를 탐지
    yolo_results = yolo_model.track(frame, persist=True, tracker='botsort.yaml', classes=[0], conf=0.4, verbose=False)

    current_track_ids = set()
    # 추적 결과 유무 확인
    if yolo_results[0].boxes.id is not None:
        # 추적된 객체들의 바운딩 박스와 ID를 가져옴
        boxes = yolo_results[0].boxes.xyxy.cpu()
        track_ids = yolo_results[0].boxes.id.int().cpu().tolist()
        confs = yolo_results[0].boxes.conf.cpu().tolist()

        # 신뢰도 순으로 정렬
        sorted_indices = sorted(range(len(confs)), key=lambda k: confs[k], reverse=True)
        # 목표 인원수만큼만 상위 객체를 선택
        top_indices = sorted_indices[:TARGET_PERSON_COUNT]

        filtered_boxes = [boxes[i] for i in top_indices]
        filtered_track_ids = [track_ids[i] for i in top_indices]
        current_track_ids = set(filtered_track_ids)

        # 탐지된 각 사람에 대해 반복 처리
        for box, track_id in zip(filtered_boxes, filtered_track_ids):
            # 새로운 사람이면 TrackedPerson 객체 생성
            if track_id not in tracked_skeletons:
                tracked_skeletons[track_id] = TrackedPerson(track_id)

            person = tracked_skeletons[track_id]
            person.disappeared_frames = 0   # 다시 나타났으므로 카운터 초기화

            # 바운딩 박스 좌표 추출 (x1, y1, x2, y2)
            x1, y1, x2, y2 = [int(c) for c in box]

            # --- 포즈 추정 및 칼만 필터 업데이트 ---
            # 바운딩 박스 영역 확장 (Padding)
            padding = 0.15  # 15%의 여유 공간을 줌
            box_w = x2 - x1
            box_h = y2 - y1
            # 패딩을 적용하되, 프레임 경계를 넘어가지 않도록 좌표 보정
            x1_pad = max(0, int(x1 - box_w * padding))
            y1_pad = max(0, int(y1 - box_h * padding))
            x2_pad = min(frame_width, int(x2 + box_w * padding))
            y2_pad = min(frame_height, int(y2 + box_h * padding))

            # STEP 2: 개별 영역 추출 및 포즈 추정 (Crop & MediaPipe)
            # 탐지된 사람의 영역만 잘라내기
            person_crop = frame[y1_pad:y2_pad, x1_pad:x2_pad]

            # 잘라낸 이미지가 비어있지 않은지 확인
            if person_crop.shape[0] > 0 and person_crop.shape[1] > 0:
                # MediaPipe Pose는 RGB 이미지를 입력으로 받음
                crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
                pose_results = pose.process(crop_rgb)

                # STEP 3: 좌표 변환 및 시각화
                if pose_results.pose_landmarks and len(pose_results.pose_landmarks.landmark) == 33:
                    # 원본 프레임에 YOLO 바운딩 박스 그리기
                    cv2.rectangle(frame, (x1,y1), (x2, y2), person.color, 2)
                    cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, person.color, 2)

                    crop_h, crop_w, _ = person_crop.shape

                    for i, landmark in enumerate(pose_results.pose_landmarks.landmark):
                        # 측정된 현재 위치
                        measured_x = x1_pad + landmark.x * crop_w
                        measured_y = y1_pad + landmark.y * crop_h

                        # 칼만 필터 적용
                        kf = person.kalman_filters[i]

                        # 칼만 필터의 상태가 초기화되지 않았다면 (첫 프레임), 현재 위치로 초기화
                        if np.all(kf.state[0:2] == 0):  # 첫 감지 시 초기화
                            kf.state[0] = measured_x
                            kf.state[1] = measured_y

                        # 예측 및 업데이트
                        kf.predict()
                        updated_state = kf.update(np.array([[measured_x], [measured_y]]))

                        # 업데이트된 랜드마크 정보 저장
                        person.landmarks[i] = [updated_state[0, 0], updated_state[1, 0], landmark.z, landmark.visibility]

    # STEP 2: 상태 예측 및 보간 (사라진 사람 처리)
    # 현재 프레임에서 사라진 ID들을 찾음
    disappeared_ids = set(tracked_skeletons.keys()) - current_track_ids

    for track_id in list(disappeared_ids):  # list()로 복사하여 반복 중 삭제 오류 방지
        person = tracked_skeletons[track_id]
        person.disappeared_frames += 1

        # 너무 오래 사라졌으면 목록에서 최종 삭제
        if person.disappeared_frames > MAX_DISAPPEARED_FRAMES:
            del tracked_skeletons[track_id]
            continue

        # 예측된 위치로 희미한 스켈레톤 그리기
        for i in range(33):
            kf = person.kalman_filters[i]
            predicted_state = kf.predict()  # 측정값 없이 예측만 수행
            person.landmarks[i, 0] = predicted_state[0, 0]
            person.landmarks[i, 1] = predicted_state[1, 0]

        # 희미한 색상으로 ID 표시
        faint_color = tuple(c // 2 for c in person.color)
        cv2.putText(frame, f"ID: {track_id} (Lost)", (int(person.landmarks[0,0]), int(person.landmarks[0,1]) - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, faint_color, 2)

    # STEP 3: 최종 데이터 저장 및 시각화 (모든 추적된 스켈레톤 그리기)
    for track_id, person in tracked_skeletons.items():
        # 마지막으로 저장/예측된 랜드마크로 스켈레톤 그리기
        landmarks_to_process = person.landmarks
        color = person.color if person.disappeared_frames == 0 else tuple(c // 2 for c in person.color)

        # CSV 데이터 저장
        for i, landmark in enumerate(landmarks_to_process):
            row_data = [
                frame_counter,
                track_id,
                i,
                landmark[0],    # x
                landmark[1],    # y
                landmark[2],    # z
                landmark[3],    # visibility
            ]
            csv_writer.writerow(row_data)

        for landmark in landmarks_to_process:
            if landmark[3] > 0.5:
                cv2.circle(frame, (int(landmark[0]), int(landmark[1])), 3, color, -1)

        connections = mp_pose.POSE_CONNECTIONS
        for connection in connections:
            start_idx, end_idx = connection
            if landmarks_to_process[start_idx, 3] > 0.5 and landmarks_to_process[end_idx, 3] > 0.5:
                start_point = (int(landmarks_to_process[start_idx, 0]), int(landmarks_to_process[start_idx, 1]))
                end_point = (int(landmarks_to_process[end_idx, 0]), int(landmarks_to_process[end_idx, 1]))
                cv2.line(frame, start_point, end_point, color, 2)


    #처리된 프레임을 결과 동영상에 쓰기
    out.write(frame)
    # 실시간으로 처리 과정 보기
    cv2.imshow('Top-Down Pose Estimation', frame)

    frame_counter += 1 # 💡 프레임 카운터 증가
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --------------------------
# 종료 처리
# --------------------------
csv_file.close()
cap.release()
out.release()
cv2.destroyAllWindows()
pose.close()

print(f"처리 완료: '{output_video_path}'에 저장되었습니다.")


스켈레톤 추출을 시작합니다. 결과는 동영상 파일로 저장됩니다.
처리 완료: 'aespa_test_output.mp4'에 저장되었습니다.


In [ ]:
!pip install pandas openpyxl

In [ ]:
# 롱 포멧 방식의 결과를 사람의 가독성을 위해 와이드 포멧으로 변경하는 코드
import pandas as pd
import os
import mediapipe as mp

# --- 설정 ---
# 변환할 원본 데이터 파일 경로
input_csv_path = 'aespa_test_skeletons_data.csv' 

# 최종적으로 생성될 엑셀 보고서 파일 경로
output_excel_path = input_csv_path.replace('_skeletons_data.csv', '_skeletons_report.xlsx')

# MediaPipe의 PoseLandmark enum을 사용하여, ID를 실제 관절 이름으로 매핑합니다.
landmark_names = [name.name for name in mp.solutions.pose.PoseLandmark]

# --- 데이터 변환 로직 ---
def convert_long_to_wide_excel(csv_path, excel_path):
    """
    '롱 포맷' CSV 데이터를 읽어, track_id별로 시트를 나눈 '와이드 포맷' 엑셀 파일로 변환합니다.
    이때, 열을 '관절 부위' 중심으로 정렬합니다. (예: x_NOSE, y_NOSE, z_NOSE, v_NOSE, ...)
    """
    if not os.path.exists(csv_path):
        print(f"오류: 원본 데이터 파일 '{csv_path}'를 찾을 수 없습니다.")
        return

    print(f"'{csv_path}' 파일을 읽는 중입니다...")
    df_long = pd.read_csv(csv_path)
    
    track_ids = df_long['track_id'].unique()
    
    print(f"총 {len(track_ids)}개의 고유한 Track ID를 발견했습니다: {sorted(track_ids)}")
    print(f"'{excel_path}' 엑셀 파일 생성을 시작합니다...")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for track_id in sorted(track_ids):
            print(f"  - Track ID: {track_id} 시트를 처리하는 중...")
            
            df_person = df_long[df_long['track_id'] == track_id]
            
            df_wide = df_person.pivot_table(
                index='frame', 
                columns='landmark_id', 
                values=['x', 'y', 'z', 'visibility']
            )
            
            # 💡 [개선] 열 순서를 관절 중심으로 재정렬합니다.
            # 1. 원하는 순서대로 컬럼 리스트를 새로 생성합니다.
            ordered_columns_tuples = []
            for i in range(len(landmark_names)): # 0부터 32까지 관절 ID 순회
                for axis in ['x', 'y', 'z', 'visibility']: # 각 관절에 대해 x, y, z, v 순서로 추가
                    ordered_columns_tuples.append((axis, i))

            # 2. 생성된 순서대로 DataFrame의 열을 재정렬합니다.
            df_wide = df_wide[ordered_columns_tuples]

            # 3. 재정렬된 컬럼의 이름을 직관적으로 변경합니다.
            new_columns = []
            for axis, idx in df_wide.columns:
                landmark_name = landmark_names[idx]
                axis_name = 'v' if axis == 'visibility' else axis
                new_columns.append(f'{axis_name}_{landmark_name}')
            
            df_wide.columns = new_columns
            
            df_wide = df_wide.sort_index()

            sheet_name = f'Track_ID_{track_id}'
            df_wide.to_excel(writer, sheet_name=sheet_name)

    print("\n변환 완료!")
    print(f"최종 보고서가 '{excel_path}' 경로에 성공적으로 저장되었습니다.")


# --- 스크립트 실행 ---
if __name__ == "__main__":
    convert_long_to_wide_excel(input_csv_path, output_excel_path)

'aespa_test_skeletons_data.csv' 파일을 읽는 중입니다...
총 14개의 고유한 Track ID를 발견했습니다: [1, 2, 3, 4, 5, 9, 10, 11, 13, 14, 15, 16, 18, 19]
'aespa_test_skeletons_report_final.xlsx' 엑셀 파일 생성을 시작합니다...
  - Track ID: 1 시트를 처리하는 중...
  - Track ID: 2 시트를 처리하는 중...
  - Track ID: 3 시트를 처리하는 중...
  - Track ID: 4 시트를 처리하는 중...
  - Track ID: 5 시트를 처리하는 중...
  - Track ID: 9 시트를 처리하는 중...
  - Track ID: 10 시트를 처리하는 중...
  - Track ID: 11 시트를 처리하는 중...
  - Track ID: 13 시트를 처리하는 중...
  - Track ID: 14 시트를 처리하는 중...
  - Track ID: 15 시트를 처리하는 중...
  - Track ID: 16 시트를 처리하는 중...
  - Track ID: 18 시트를 처리하는 중...
  - Track ID: 19 시트를 처리하는 중...

변환 완료!
최종 보고서가 'aespa_test_skeletons_report_final.xlsx' 경로에 성공적으로 저장되었습니다.
